In [1]:
import os
base_path = os.path.dirname(os.path.abspath(""))
base_path = base_path + "/headers/"
print(base_path)
header_path = base_path + 'header_data_treatment.ipynb'
%run $header_path

C:\Users\raphael.wargon\Documents\GitHub\sorting_univ\script/headers/
initalizing spark nlp session w gpu


In [2]:
int_save_path = save_path + 'intermediary\\'

In [3]:
int_save_path

'D:\\panel_fr_res\\data\\intermediary\\'

# Load data

In [4]:
works_au_af = (spark.read.format('parquet').load('file:\\' + main_path_openalex + 'works_au_af.parquet')
               .filter( func.col('publication_year').between(1950,2020) )
               .select('work_id','publication_year','author_id','source_id','citations','primary_topic','inst_id','country')

              )

In [5]:
journals_ranking = (spark.read.format('parquet').load('file:\\' + main_path_openalex + 'journals_ranking_year.parquet')
                    .withColumn('source_id', func.regexp_replace(func.col('source_id'), 'https://openalex.org/', ''))
                   )
journals_ranking.show()

+-----------+----+-----------+-------------------+------------------+
|  source_id|year|source_type|    rank_source_pct|   abs_rank_source|
+-----------+----+-----------+-------------------+------------------+
| S100014387|2021|    journal|                0.0|               1.0|
| S100014455|2010|    journal| 0.8969174714671388|1275.9381443298969|
|S1000263748|2002|    journal| 0.8676467635588629| 872.2857142857142|
|S1000263748|2019|    journal| 0.8747890323638025|3337.1707317073174|
| S100028386|2005|    journal| 0.8903641881638846|             284.0|
| S100045985|1964|    journal|  0.909952606635071|              96.0|
| S100060201|2011|    journal| 0.8432467959250739|            2382.0|
| S100062655|2015|    journal| 0.9088593802466257|            1650.0|
| S100062975|2020|    journal| 0.9048618716892145| 2239.538461538462|
|  S10007838|1972|    journal|0.15706909290143697| 839.8000000000001|
|S1000839972|1979|    journal|  0.679080824088748|             387.0|
|S1000839972|2018|  

In [6]:
df_au = (spark.read.parquet('file:///' + main_path_openalex + 'authors.parquet')
                    .withColumn('author_id', func.regexp_replace(func.col('id'), 'https://openalex.org/', ''))
        )

In [4]:
budget_inst_matched = spark.read.csv('file:///' + scanR_path + '\\budget_inst_matched.csv', header = True)

# Build the raw dataset

In [7]:
inst = spark.read.parquet('file:\\' + save_path + 'intermediary\\inst_fr.parquet')

In [8]:
papers_textual_metrics = spark.read.csv('file:///' + databases_path + '/papers_textual_metrics.csv', header = True) 
papers_textual_metrics = papers_textual_metrics.withColumnRenamed('paperID', 'work_id').drop(*['n_words','n_phrases', 'has_abstract'])

In [9]:
au_inst_level_flow = (works_au_af
                      .join(journals_ranking.withColumnRenamed('year','publication_year'), on = ['source_id','publication_year'], how = 'left')

                      #.withColumn('inst_id', func.when(func.col('inst_id')=='I118618916',  "I899635006")
                      #                           .when(func.col('inst_id')=="I36085230",  "I899635006")
                      #                           .when(func.col('inst_id')=="I177483745",  "I899635006")
                      #                           .when(func.col('inst_id')=="I4210103002", "I198244214")
                      #                           .when(func.col('inst_id')=="I4210143836", "I198244214")
                      #                           .when(func.col('inst_id')=="I184646667",  "I39804081")
                      #                           .when(func.col('inst_id')=="I102197404",  "I277688954")
                      #                           .when(func.col('inst_id')=='I7171862',  "I2279609970")
                      #                           .when(func.col('inst_id')=="I59807433",  "I2279609970")
                      #                           .when(func.col('inst_id')=="I2800379142", "I4210154111") 
                      #                           .when(func.col('inst_id')=="I3123023596","I56067802")
                      #                           .when(func.col('inst_id')=="I4210095130","I208215962")
                      #                           .otherwise(func.col('inst_id')))
                      .filter(func.col('inst_id').isNotNull())
                      .join(inst.select('inst_id', func.lit(1).alias('in_inst_dataset')), on =['inst_id'], how = 'left')
                      .fillna(0, subset = ['in_inst_dataset'])
                    .withColumn('inst_id', func.when( (func.col('country')=="FR")
                                                     | (func.col('in_inst_dataset') ==1)
                                                     , func.col('inst_id')).otherwise('abroad'))
                      
                    .groupBy('author_id',func.col('publication_year').alias('year'),'work_id')
                    .agg(func.collect_set('inst_id').alias('inst_id_set_y'),
                         *[func.first(col).alias(col) for col in ['citations', 'rank_source_pct'] + [col for col in papers_textual_metrics.columns if col != 'work_id']]
                        )
                      .withColumn('coau_set', func.collect_set(func.col('author_id')).over(Window.partitionBy('work_id')))
                    .groupBy('author_id','year')
                    .agg(func.array_compact(func.flatten(func.collect_set('inst_id_set_y'))).alias('inst_id_set_y'),
                         func.array_compact(func.flatten(func.collect_set('coau_set'))).alias('coau_set'),
                         func.countDistinct('work_id').alias('publications'),
                         func.sum('citations').alias('citations'),
                         func.mean('rank_source_pct').alias('avg_rank_source_pct'), 
                         func.sum('rank_source_pct').alias('rank_w_publications'),
                         func.sum((func.col('rank_source_pct')<= 0.5).cast('int')).alias('nr_source_btm_50pct_raw'),
                         func.sum((func.col('rank_source_pct').between(0.5,0.9)).cast('int')).alias('nr_source_mid_40pct_raw'),
                         func.sum((func.col('rank_source_pct')>= 0.8).cast('int')).alias('nr_source_top_20pct_raw'),
                         func.sum((func.col('rank_source_pct')>= 0.9).cast('int')).alias('nr_source_top_10pct_raw'),
                         func.sum((func.col('rank_source_pct')>= 0.95).cast('int')).alias('nr_source_top_5pct_raw'),

                        )

                    #.withColumn('inst_id_set_last_y', func.lag(func.col('inst_id_set_y')).over(Window.partitionBy('author_id').orderBy('year')))
                    #.withColumn('inst_id_receiver', func.explode(func.col('inst_id_set_y')))
                    #.withColumn('inst_id_sender',  func.explode_outer(func.col('inst_id_set_last_y')))
                    #.select('inst_id_receiver','inst_id_sender', 'author_id','year')
                   )

In [10]:
au_inst_level_flow.write.mode('overwrite').parquet('file:\\' + save_path + 'au_inst_level.parquet')

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "C:\ProgramData\anaconda3\envs\pyspark_env\lib\site-packages\py4j\clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "C:\ProgramData\anaconda3\envs\pyspark_env\lib\socket.py", line 716, in readinto
    return self._sock.recv_into(b)
ConnectionResetError: [WinError 10054] Une connexion existante a dû être fermée par l’hôte distant

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\ProgramData\anaconda3\envs\pyspark_env\lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "C:\ProgramData\anaconda3\envs\pyspark_env\lib\site-packages\py4j\clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


Py4JError: An error occurred while calling o288.parquet

# Add inst-level info

In [9]:
au_inst_level_flow = spark.read.parquet('file:\\' + save_path + "intermediary\\" + 'au_inst_level.parquet')

In [10]:
au_inst_level_flow = (au_inst_level_flow
                    .withColumn('inst_id_set_last_y', func.lag(func.col('inst_id_set_y')).over(Window.partitionBy('author_id').orderBy('year')))
                    .withColumn('inst_id_receiver', func.explode(func.col('inst_id_set_y')))
                    .withColumn('inst_id_sender',  func.explode_outer(func.col('inst_id_set_last_y')))
                    .select('inst_id_receiver','inst_id_sender', 'author_id','year')
                    .distinct()

                     )
au_inst_level_flow.printSchema()

root
 |-- inst_id_receiver: string (nullable = true)
 |-- inst_id_sender: string (nullable = true)
 |-- author_id: string (nullable = true)
 |-- year: integer (nullable = true)



In [11]:
au_inst_level_flow.cache()
au_inst_level_flow.show()

+----------------+--------------+-----------+----+
|inst_id_receiver|inst_id_sender|  author_id|year|
+----------------+--------------+-----------+----+
|          abroad|        abroad|A5000000266|2018|
|          abroad|        abroad|A5000000266|2019|
|          abroad|        abroad|A5000002279|2004|
|          abroad|        abroad|A5000002463|2002|
|          abroad|        abroad|A5000005204|1997|
|          abroad|        abroad|A5000005204|1998|
|          abroad|        abroad|A5000005631|2018|
|          abroad|        abroad|A5000005751|1998|
|          abroad|        abroad|A5000005751|2008|
|          abroad|        abroad|A5000005751|2009|
|          abroad|        abroad|A5000005751|2011|
|          abroad|        abroad|A5000005751|2012|
|          abroad|        abroad|A5000005751|2013|
|          abroad|        abroad|A5000005751|2014|
|          abroad|        abroad|A5000005751|2016|
|          abroad|        abroad|A5000005751|2018|
|          abroad|        abroa

In [12]:
au_inst_level_flow.count()

130383692

In [13]:
au_inst_level_flow.filter( ~ ( 
                                    (func.col('inst_id_sender').isin(['abroad']))
                                    &  (func.col('inst_id_receiver').isin(['abroad']))
                                )).count()

27384454

In [14]:
au_inst_level_flow_filtering = (au_inst_level_flow
                                .filter( ~ ( 
                                    (func.col('inst_id_sender').isin(['abroad']))
                                    &  (func.col('inst_id_receiver').isin(['abroad']))
                                ))
                                .join(inst
                                        .select(func.col('inst_id').alias('inst_id_receiver'), 
                                                func.col('parent.parent_id').alias('parent_id_r'))
                                        , on = 'inst_id_receiver', how ='left')
                                .join(inst
                                        .select(func.col('inst_id').alias('inst_id_sender'), 
                                                func.col('parent.parent_id').alias('parent_id_s'))
                                        , on = 'inst_id_sender', how ='left')
            
                                .withColumn('parent_id_r', func.array_except(func.col('parent_id_r'), func.array(func.col('inst_id_receiver'))))
                                .withColumn('parent_id_s', func.array_except(func.col('parent_id_s'), func.array(func.col('inst_id_sender'))))
                                .groupBy('author_id','year')
                                .agg(func.collect_set(func.col('inst_id_receiver')).alias('inst_id_receiver'),
                                     func.collect_set(func.col('inst_id_sender')).alias('inst_id_sender'),
                                     func.array_except(func.transform(func.array_distinct(func.flatten(func.collect_set(func.col('parent_id_r')))), lambda x: func.regexp_replace(x, 'https://openalex.org/', ''))
                                                                                 , func.array(func.lit(None))).alias('all_parent_ids_r'),
                                     
                                     func.array_except(func.transform(func.array_distinct(func.flatten(func.collect_set(func.col('parent_id_s')))), lambda x: func.regexp_replace(x, 'https://openalex.org/', ''))
                                                                                 , func.array(func.lit(None))).alias('all_parent_ids_s')
                               )
                                .withColumn('inst_id_receiver', func.explode(func.array_except(func.col('inst_id_receiver'), func.col('all_parent_ids_r'))))
                                .withColumn('inst_set_sender', func.col('inst_id_sender'))
                                .withColumn('inst_id_sender',   func.explode_outer(func.array_except(func.col('inst_id_sender'), func.col('all_parent_ids_s'))))

                                .select('inst_id_receiver','inst_id_sender', 'inst_set_sender', 'author_id','year')
                               )

In [15]:
au_inst_level_flow_filtering.cache()
au_inst_level_flow_filtering.show()

+----------------+--------------+--------------------+-----------+----+
|inst_id_receiver|inst_id_sender|     inst_set_sender|  author_id|year|
+----------------+--------------+--------------------+-----------+----+
|     I4210098836|          NULL|                  []|A5000016899|1960|
|     I4210152651|   I4210098836|       [I4210098836]|A5000016899|1964|
|     I4210152651|   I4210152651|       [I4210152651]|A5000016899|1965|
|     I4210152651|   I4210152651|[I4210152651, I10...|A5000016899|1966|
|     I3020098449|   I4210152651|[I4210152651, I10...|A5000016899|1966|
|     I4210152651|   I4210152651|[I4210152651, I30...|A5000016899|1967|
|     I4210152651|   I3020098449|[I4210152651, I30...|A5000016899|1967|
|      I100532134|   I4210152651|[I4210152651, I10...|A5000016899|1968|
|     I4210152651|    I100532134|        [I100532134]|A5000016899|1969|
|     I4210152651|   I4210152651|       [I4210152651]|A5000016899|1970|
|     I4210144981|   I4210152651|       [I4210152651]|A500001689

In [16]:
au_inst_level_flow_filtering.count()

17431720

In [19]:
au_inst_level_flow_filtering.filter(func.col('inst_id_receiver')=="I1294671590").count()

892961

In [20]:
au_inst_level_flow.filter(func.col('inst_id_receiver')=="I1294671590").count()

2886288

In [17]:
au_inst_level_flow.repartition('year')
au_inst_level_flow_filtering.repartition('year')

DataFrame[inst_id_receiver: string, inst_id_sender: string, inst_set_sender: array<string>, author_id: string, year: int]

In [18]:
weights_by_au_year = (au_inst_level_flow_filtering
                      .groupBy('author_id','year')
                      .count()
                      .select('author_id','year', (1/func.col('count')).alias('weights'))
                     )
weights_by_au_year.cache()
weights_by_au_year.show()

+-----------+----+------------------+
|  author_id|year|           weights|
+-----------+----+------------------+
|A5000016899|1960|               1.0|
|A5000016899|1964|               1.0|
|A5000016899|1965|               1.0|
|A5000016899|1966|               0.5|
|A5000016899|1967|               0.5|
|A5000016899|1968|               1.0|
|A5000016899|1969|               1.0|
|A5000016899|1970|               1.0|
|A5000016899|1972|               1.0|
|A5000016899|2009|0.3333333333333333|
|A5000023309|2017|               1.0|
|A5000087584|2018|0.3333333333333333|
|A5000139008|2020|               1.0|
|A5000139201|2011|               0.5|
|A5000139201|2013|              0.25|
|A5000139428|1998|               1.0|
|A5000156980|2000|               1.0|
|A5000156980|2004|               1.0|
|A5000157973|2003|               1.0|
|A5000157973|2005|               0.5|
+-----------+----+------------------+
only showing top 20 rows



In [23]:
au_inst_level_flow.select('author_id').distinct().count()

30301591

In [24]:
au_inst_level_flow_filtering.select('author_id').distinct().count()

1553692

In [25]:
inst.select('secteur').distinct().show()

+---------+
|  secteur|
+---------+
|   public|
|    privé|
|undefined|
+---------+



# author-level info

In [19]:
au_inst_level_flow_init = spark.read.parquet('file:\\' + save_path + "intermediary\\" + 'au_inst_level.parquet')

In [17]:
au_inst_level_flow_init.show()

+-----------+----+-------------+--------------------+------------+---------+-------------------+-------------------+-----------------------+-----------------------+-----------------------+-----------------------+----------------------+
|  author_id|year|inst_id_set_y|            coau_set|publications|citations|avg_rank_source_pct|rank_w_publications|nr_source_btm_50pct_raw|nr_source_mid_40pct_raw|nr_source_top_20pct_raw|nr_source_top_10pct_raw|nr_source_top_5pct_raw|
+-----------+----+-------------+--------------------+------------+---------+-------------------+-------------------+-----------------------+-----------------------+-----------------------+-----------------------+----------------------+
|A5000000264|2000|[I4210114068]|[A5069745915, A50...|           2|       52| 0.9302471721826561| 0.9302471721826561|                      0|                      0|                      1|                      1|                     0|
|A5000000717|2012|     [abroad]|[A5000000717, A51...|   

In [18]:
au_level_info = (au_inst_level_flow_init
                 #.join(au_to_keep, on= 'author_id', how ='inner')
                 .groupBy('author_id')
                 .agg(func.min('year').alias('entry_year'),
                      func.max('year').alias('exit_year'))
                 .join(au_inst_level_flow
                       .groupBy('author_id', func.col('year').alias("entry_year"))
                       .agg(func.array_sort(func.collect_set(func.col('inst_id_receiver'))).alias('entry_inst_id')
                           )
                       , on = ['author_id', 'entry_year'], how = 'left')
                )

In [19]:
au_level_info.write.mode('overwrite').parquet('file:\\' + save_path + 'au_level_for_inst_dataset.parquet')

In [20]:
au_level_info = spark.read.parquet('file:\\' + save_path + 'intermediary\\au_level_for_inst_dataset.parquet')

In [21]:
gendersort_path =  os.path.dirname(os.path.abspath("")) + "\\additional_material\\Wiki_Gendersort.py"
%run $gendersort_path
udf_wiki_gendersort = func.udf(wiki_gendersort().assign,StringType())


In [22]:
au_level_info = (au_level_info
                 .join(df_au
                       .select('author_id', 
                               func.col('display_name').alias('name'))
                       .withColumn('author_first_name', func.element_at(func.split(func.col('name'), ' '), 1))
                       .withColumn('gender', udf_wiki_gendersort(func.col('author_first_name')))
                       .select('author_id','gender'), on = 'author_id', how = 'left')
                )

                       

In [23]:
au_level_info.count()

30301591

In [24]:
au_inst_level_plus_info = (au_inst_level_flow_filtering
                           .join(au_level_info, on = ['author_id'], how =  'left'))
au_inst_level_plus_info.cache()
au_inst_level_plus_info.count()

17431720

In [25]:
au_inst_level_plus_info.show()

+-----------+----------------+--------------+--------------------+----+----------+---------+--------------------+------+
|  author_id|inst_id_receiver|inst_id_sender|     inst_set_sender|year|entry_year|exit_year|       entry_inst_id|gender|
+-----------+----------------+--------------+--------------------+----+----------+---------+--------------------+------+
|A5000016899|     I4210098836|          NULL|                  []|1960|      1960|     2009|       [I4210098836]|  NULL|
|A5000016899|     I4210152651|   I4210098836|       [I4210098836]|1964|      1960|     2009|       [I4210098836]|  NULL|
|A5000016899|     I4210152651|   I4210152651|       [I4210152651]|1965|      1960|     2009|       [I4210098836]|  NULL|
|A5000016899|     I4210152651|   I4210152651|[I4210152651, I10...|1966|      1960|     2009|       [I4210098836]|  NULL|
|A5000016899|     I3020098449|   I4210152651|[I4210152651, I10...|1966|      1960|     2009|       [I4210098836]|  NULL|
|A5000016899|     I4210152651|  

In [26]:
au_inst_level_plus_info.filter(func.col('author_id')=="A5053834730").filter(func.col('inst_id_sender')=="I187986737").show()

+-----------+----------------+--------------+--------------------+----+----------+---------+-------------+
|  author_id|inst_id_receiver|inst_id_sender|     inst_set_sender|year|entry_year|exit_year|entry_inst_id|
+-----------+----------------+--------------+--------------------+----+----------+---------+-------------+
|A5053834730|       I90669466|    I187986737|[I90669466, I1294...|2001|      1972|     2020|[I4210126650]|
|A5053834730|          abroad|    I187986737|[I90669466, I1294...|2001|      1972|     2020|[I4210126650]|
|A5053834730|      I187986737|    I187986737|[I90669466, I1294...|2001|      1972|     2020|[I4210126650]|
|A5053834730|     I1294671590|    I187986737|[I90669466, I1294...|2001|      1972|     2020|[I4210126650]|
|A5053834730|       I90669466|    I187986737|[I90669466, I1879...|2002|      1972|     2020|[I4210126650]|
|A5053834730|          abroad|    I187986737|[I90669466, I1879...|2002|      1972|     2020|[I4210126650]|
|A5053834730|     I1294671590|    I18

# institution-to-institution-level raw flows

In [22]:
pre_inst_flow = (au_inst_level_plus_info
                    .filter(func.col('year').between(1980,2020))
                  .withColumn('stayer', func.array_contains(func.col('inst_set_sender'), func.col('inst_id_receiver')).cast('int'))

                  .withColumn('junior', (func.col('year')-func.col('entry_year')<=5).cast('int'))
                  .withColumn('senior', (func.col('year')-func.col('entry_year')>=15).cast('int'))
                  .withColumn('medium', ((func.col('year')-func.col('entry_year')>5) & (func.col('year')-func.col('entry_year')<15)) .cast('int'))

                  .withColumn('own_entrant_r', (func.array_contains(func.col('entry_inst_id'), func.col('inst_id_receiver')) ).cast('int'))
                  .withColumn('own_entrant_s', (func.array_contains(func.col('entry_inst_id'), func.col('inst_id_sender')) ).cast('int'))

                  .withColumn('foreign_entrant', (func.array_contains(func.col('entry_inst_id'),('abroad')) ).cast('int'))
                 .join(weights_by_au_year, on = ['author_id','year'], how = 'left')
                  )


In [34]:
pre_inst_flow.filter(func.col('year')==func.col('exit_year')).show()

+-----------+----+----------------+--------------+--------------------+----------+---------+--------------------+------+------+------+------+------+-------------+-------------+---------------+------------------+
|  author_id|year|inst_id_receiver|inst_id_sender|     inst_set_sender|entry_year|exit_year|       entry_inst_id|gender|stayer|junior|senior|medium|own_entrant_r|own_entrant_s|foreign_entrant|           weights|
+-----------+----+----------------+--------------+--------------------+----------+---------+--------------------+------+------+------+------+------+-------------+-------------+---------------+------------------+
|A5000002348|2017|     I4210135719|          NULL|                  []|      2017|     2017|       [I4210135719]|     F|     0|     1|     0|     0|            1|         NULL|              0|               1.0|
|A5000003497|2019|     I2802759292|     I37553959|         [I37553959]|      2014|     2019|            [abroad]|   UNK|     0|     1|     0|     0|    

In [24]:
pre_inst_flow.write.mode('overwrite').parquet('file:\\' + save_path + 'intermediary\\pre_inst_flow.parquet')

In [27]:
pre_inst_flow = spark.read.parquet('file:\\' + save_path + 'intermediary\\pre_inst_flow.parquet')

In [27]:
au_prod_measures = (au_inst_level_flow_init
                    .withColumn('entry_year', func.min(func.col('year')).over(Window.partitionBy('author_id')))
                    .groupBy('author_id','entry_year')
                    .agg(func.sum(func.col('citations')*(func.col('year')<= func.col('entry_year')+2).cast('int')).alias('cit_2y'),
                         func.sum('citations').alias('cit'))
                    .withColumn('pct_cit_2y', func.percent_rank().over(Window.partitionBy().orderBy('cit_2y')))
                    .withColumn('pct_cit', func.percent_rank().over(Window.partitionBy().orderBy('cit')))
                    .select('author_id', 'pct_cit','pct_cit_2y')
                   )
au_prod_measures.cache()
au_prod_measures.show()

+-----------+-------+----------+
|  author_id|pct_cit|pct_cit_2y|
+-----------+-------+----------+
|A5000000131|    0.0|       0.0|
|A5000000356|    0.0|       0.0|
|A5000001855|    0.0|       0.0|
|A5000002279|    0.0|       0.0|
|A5000003910|    0.0|       0.0|
|A5000004249|    0.0|       0.0|
|A5000004830|    0.0|       0.0|
|A5000005590|    0.0|       0.0|
|A5000007781|    0.0|       0.0|
|A5000008373|    0.0|       0.0|
|A5000008796|    0.0|       0.0|
|A5000009532|    0.0|       0.0|
|A5000009924|    0.0|       0.0|
|A5000010796|    0.0|       0.0|
|A5000016284|    0.0|       0.0|
|A5000017023|    0.0|       0.0|
|A5000017858|    0.0|       0.0|
|A5000017860|    0.0|       0.0|
|A5000018643|    0.0|       0.0|
|A5000019154|    0.0|       0.0|
+-----------+-------+----------+
only showing top 20 rows



In [35]:
pre_inst_flow = (pre_inst_flow
                 .join(df_au.select('author_id', func.element_at(func.col('topics'), 1).alias('topic'), 'summary_stats.h_index')
                       .select('author_id',  func.regexp_replace(func.col('topic.domain.id'), 'https://openalex.org/domains/', '').alias("domain"),
                              'h_index')
                       , on = 'author_id', how = 'left')
                 .join(au_prod_measures
                       .select('author_id',
                               *[func.when(func.col(col)<=0.25, 'Q1').when(func.col(col)<=0.5, 'Q2').when(func.col(col)<=0.75, 'Q3').otherwise('Q4').alias(col.replace('pct','quart'))
                                 for col in ['pct_cit','pct_cit_2y']])
                               
                               , on = 'author_id', how = 'left')
                )

In [36]:
pre_inst_flow

DataFrame[author_id: string, year: int, inst_id_receiver: string, inst_id_sender: string, inst_set_sender: array<string>, entry_year: int, exit_year: int, entry_inst_id: array<string>, gender: string, stayer: int, junior: int, senior: int, medium: int, own_entrant_r: int, own_entrant_s: int, foreign_entrant: int, weights: double, domain: string, h_index: bigint, quart_cit: string, quart_cit_2y: string]

In [36]:
pre_inst_flow.count() #19865458

17086902

## at the merged institution level

In [37]:
#pre_inst_flow.filter(func.col('inst_id_sender')=="abroad").show()

In [38]:
#inst.filter(func.col('merged_inst_id').isNull()).show()

In [24]:
from itertools import product

inst_level_flows_merged = (pre_inst_flow
                           .withColumn('inst_id_sender', func.when((func.col('inst_id_sender').isNull()) & 
                                                                    (func.col('year')==func.col('entry_year')), 'entry')
                                                                   .otherwise(func.col('inst_id_sender')))
                           .withColumn('inst_id_receiver', func.when( #(func.col('inst_id_sender') == func.col('inst_id_receiver')) & 
                                                                    (func.col('year')==func.col('exit_year')), 'exit')
                                                                   .otherwise(func.col('inst_id_receiver')))

                      .join(inst
                            .select(func.col('inst_id').alias('inst_id_receiver'), 
                                    func.col('merged_inst_id').alias('merged_inst_id_r')
                                   )
                            , on = 'inst_id_receiver', how ='left')
                      .join(inst
                            .select(func.col('inst_id').alias('inst_id_sender'), 
                                    func.col('merged_inst_id').alias('merged_inst_id_s')
                                   )
                            , on = 'inst_id_sender', how ='left')
                    .withColumn('merged_inst_id_r', func.when(func.col('merged_inst_id_r').isNull(), func.col('inst_id_receiver')).otherwise(func.col('merged_inst_id_r')) )
                    .withColumn('merged_inst_id_s', func.when(func.col('merged_inst_id_s').isNull(), func.col('inst_id_sender')).otherwise(func.col('merged_inst_id_s')) )

                           .withColumn('gender', func.when(func.col('gender')=="M", "men")
                                       .when(func.col('gender')=="F", 'women')
                                      )
                    .groupBy('merged_inst_id_r','merged_inst_id_s', 'domain', 'year')
                      .agg(func.countDistinct('author_id').alias('total'),
                           func.sum('stayer').alias('stayers'),
                           func.sum(1-func.col('stayer')).alias('movers'),
                        
                           func.sum('weights').alias('total_w'),
                           func.sum(func.col('weights')*func.col('stayer')).alias('stayers_w'),
                           func.sum(func.col('weights')*(1-func.col('stayer'))).alias('movers_w'),


                           func.sum( (func.col('year') == func.col('exit_year')).cast('int') ).alias('exits'),
                           func.sum(func.col('weights')*(func.col('year') == func.col('exit_year')).cast('int') ).alias('exits_w'),


                           *[func.sum(category).alias('total_' + category )
                             for category in ['junior','senior','medium', 'own_entrant_r','own_entrant_s', 'foreign_entrant']
                           ],

                           *[func.sum(func.col('stayer')*func.col(category)).alias('stayers_' + category) for category in 
                             ['junior','senior','medium', 'own_entrant_r','own_entrant_s', 'foreign_entrant']
                           ],

                           *[func.sum( (1-func.col('stayer'))*func.col(category)).alias('movers_' + category) for category in ['junior','senior','medium', 'own_entrant_r','own_entrant_s', 'foreign_entrant']
                           ],
                           
                           #*[func.sum(func.col(category)*(func.col('exit_year')==func.col('year')).cast('int') ).alias('exits_' + category)for category in 
                           #  ['junior','senior','medium', 'own_entrant_r','own_entrant_s', 'foreign_entrant']
                           #],

                        
                           *[func.sum(func.col('weights')*func.col(category)).alias('total_w_' + category)for category in 
                             ['junior','senior','medium', 'own_entrant_r','own_entrant_s', 'foreign_entrant']
                           ],

                           #*[func.sum(func.col('weights')*func.col(category)*(func.col('exit_year')==func.col('year')).cast('int') ).alias('exits_w_' + category)for category in 
                           #  ['junior','senior','medium', 'own_entrant_r','own_entrant_s', 'foreign_entrant']
                           #],

                           *[func.sum(func.col('weights')*func.col('stayer')*func.col(category)).alias('stayers_w_' + category) for category in ['junior','senior','medium', 'own_entrant_r','own_entrant_s', 'foreign_entrant']
                           ],
                           *[func.sum(func.col('weights')*(1-func.col('stayer'))*func.col(category)).alias('movers_w_' + category) for category in ['junior','senior','medium', 'own_entrant_r','own_entrant_s', 'foreign_entrant']
                           ],

                           *[func.sum((func.col(prod)==quartile).cast('int') ).alias('total_' + quartile + '_' + prod.replace('quart', '') )
                             for quartile, prod in product(['Q1','Q2','Q3','Q4'], ['quart_cit','quart_cit_2y'])
                           ],
                           *[func.sum((1-func.col('stayer'))*(func.col(prod)==quartile).cast('int') ).alias('movers_' + quartile + '_' + prod.replace('quart', '') )
                             for quartile, prod in product(['Q1','Q2','Q3','Q4'], ['quart_cit','quart_cit_2y'])
                           ],
                           #*[func.sum((func.col('year')==func.col('exit_year')).cast('int')*(func.col(prod)==quartile).cast('int') ).alias('exits_' + quartile + '_' + prod.replace('quart', '') )
                           #  for quartile, prod in product(['Q1','Q2','Q3','Q4'], ['quart_cit','quart_cit_2y'])
                           #],

                           *[func.sum(func.col('weights')*(func.col(prod)==quartile).cast('int') ).alias('total_w_' + quartile + '_' + prod.replace('quart', '') )
                             for quartile, prod in product(['Q1','Q2','Q3','Q4'], ['quart_cit','quart_cit_2y'])
                           ],
                           *[func.sum(func.col('weights')*(1-func.col('stayer'))*(func.col(prod)==quartile).cast('int') ).alias('movers_w_' + quartile + '_' + prod.replace('quart', '') )
                             for quartile, prod in product(['Q1','Q2','Q3','Q4'], ['quart_cit','quart_cit_2y'])
                           ],
                           #*[func.sum(func.col('weights')*(func.col('year')==func.col('exit_year')).cast('int')*(func.col(prod)==quartile).cast('int') ).alias('exits_w_' + quartile + '_' + prod.replace('quart', '') )
                           #  for quartile, prod in product(['Q1','Q2','Q3','Q4'], ['quart_cit','quart_cit_2y'])
                           #],
#
                           
                           *[func.sum((func.col('gender')==gender).cast('int') ).alias('total_' + gender )
                             for gender in ['men','women']
                           ],
                           *[func.sum((1-func.col('stayer'))*(func.col("gender")==gender).cast('int') ).alias('movers_' + gender  )
                             for gender in ['men','women']
                           ],

                           *[func.sum(func.col('weights')*(func.col('gender')==gender).cast('int') ).alias('total_w_' + gender  )
                             for gender in ['men','women']
                           ],
                           *[func.sum(func.col('weights')*(1-func.col('stayer'))*(func.col("gender")==gender).cast('int') ).alias('movers_w_' + gender )
                             for gender in ['men','women']
                           ],
                           
                          # *[func.sum((func.col("gender")==gender).cast('int')*(func.col('exit_year')==func.col('year')).cast('int') ).alias('exits_' + gender )
                          #   for gender in ['men','women']
                          # ],
#
                          # *[func.sum(func.col('weights')*(func.col("gender")==gender).cast('int')*(func.col('exit_year')==func.col('year')).cast('int') ).alias('exits_w_' + gender ) 
                          #   for gender in ['men','women']
                          # ],

                          )
                    #.withColumn('size_r', func.max(func.col('stayers') * (func.col('inst_id_receiver')==func.col('inst_id_sender')).cast('int'))
                    #
                    #            .over(Window.partitionBy('inst_id_receiver','year')))
                    #
                    #.withColumn('size_s', func.max(func.col('stayers') * (func.col('inst_id_receiver')==func.col('inst_id_sender')).cast('int'))
                    #                  .over(Window.partitionBy('inst_id_sender','year')))
                   )


NameError: name 'pre_inst_flow' is not defined

In [ ]:
inst_level_flows_merged.cache()
inst_level_flows_merged.show()

In [53]:
inst_level_flows_merged.filter(func.col('merged_inst_id_r')=="exit").show()

+----------------+----------------+------+----+-----+-------+------+------------------+------------------+-------------------+-----+------------------+------------+------------+------------+-------------------+-------------------+---------------------+--------------+--------------+--------------+---------------------+---------------------+-----------------------+-------------+-------------+-------------+--------------------+--------------------+----------------------+------------------+------------------+------------------+---------------------+---------------------+-----------------------+------------------+------------------+------------------+-----------------------+-----------------------+-------------------------+------------------+-------------------+------------------+----------------------+----------------------+------------------------+-------------+----------------+-------------+----------------+-------------+----------------+-------------+----------------+--------------+----

In [54]:
inst_level_flows_merged.count() #2889363 # 11_06_26 :2701806

2701806

In [42]:
#inst_level_flows_merged.summary().show()

In [56]:
all_y_dyads = (inst_level_flows_merged
               .groupBy('merged_inst_id_r','merged_inst_id_s', 'domain')
               .agg(func.min('year').alias('entry_year_dyad'),
                   func.sequence(func.min('year'), func.lit(2020)).alias('year'))
               .withColumn('year', func.explode(func.col('year')))
              )
all_y_dyads.cache()
all_y_dyads.show()

+----------------+----------------+------+---------------+----+
|merged_inst_id_r|merged_inst_id_s|domain|entry_year_dyad|year|
+----------------+----------------+------+---------------+----+
|     I4210094685|          abroad|     4|           1981|1981|
|     I4210094685|          abroad|     4|           1981|1982|
|     I4210094685|          abroad|     4|           1981|1983|
|     I4210094685|          abroad|     4|           1981|1984|
|     I4210094685|          abroad|     4|           1981|1985|
|     I4210094685|          abroad|     4|           1981|1986|
|     I4210094685|          abroad|     4|           1981|1987|
|     I4210094685|          abroad|     4|           1981|1988|
|     I4210094685|          abroad|     4|           1981|1989|
|     I4210094685|          abroad|     4|           1981|1990|
|     I4210094685|          abroad|     4|           1981|1991|
|     I4210094685|          abroad|     4|           1981|1992|
|     I4210094685|          abroad|     

In [57]:
#inst_merged_level = (inst
#                     .withColumn('merged_inst_id', func.when(func.col('merged_inst_id').isNull(), func.col('inst_id')).otherwise(func.col('merged_inst_id')))
#                     .groupBy('merged_inst_id')
#                     .agg( *[func.first(func.when(func.col('merged_inst_id')==func.col('inst_id'), func.col(col)), ignorenulls = True).alias(col)
#                             for col in inst.columns if col not in ['inst_id','merged_inst_id','acces_rce', 'date_first_idex', 'fusion_date']],
#                          *[func.min(func.when((func.col(col) >0) & (func.col(col).isNotNull()),
#                                               func.col(col).cast(StringType())).otherwise("0")).alias(col)
#                          for col in ['acces_rce','date_first_idex','fusion_date']])
#                    )
#inst_merged_level.show(truncate = False)

In [58]:
#inst.filter(func.array_contains(func.col('parent.parent_id'), "I277688954")).filter(func.col('acces_rce').isNull()).show(truncate = False)

In [60]:
panel_inst_flows_merged = (all_y_dyads
                           .join(inst_level_flows_merged
                                 .select('merged_inst_id_r','merged_inst_id_s','domain','year', *[col for col in  inst_level_flows_merged.columns if 
                                                'total' in col or 'stayer' in col or 'mover' in col or "exit" in col])
                                 , on = ['merged_inst_id_r','merged_inst_id_s', 'domain','year'], how = 'left')
                           .fillna(0, subset = [col for col in  inst_level_flows_merged.columns if 
                                                'total' in col or 'stayer' in col or 'mover' in col or "exit" in col])
                           
                           .join(inst
                                .withColumn('merged_inst_id', func.when(func.col('merged_inst_id').isNull(), func.col('inst_id'))
                                                   .otherwise(func.col('merged_inst_id')))
                                .filter(func.col('merged_inst_id')==func.col('inst_id'))
                                        .select(*[func.col(col).alias(col +'_r') for col in ['merged_inst_id','name','city','uni_pub','cnrs','type','main_topic','type_fr','secteur', 'acces_rce','date_first_idex', 'fusion_date']],
                                               func.concat_ws(';', func.transform(func.col('topic_share'), lambda x: func.to_json(x))).alias('topic_share_r'),
                                               *[ (func.col('type_fr') == type_).cast('int').alias(type_.lower().replace('é', 'e') +'_r')
                                                     for type_ in ['Université', 'École']],
                                                *[(func.col('secteur') == type_).cast('int').alias(type_.lower().replace('é', 'e') +'_r')
                                                 for type_ in ['public', 'privé']]
                                               )
                            , on = 'merged_inst_id_r', how ='left')
                      .join(inst
                            .withColumn('merged_inst_id', func.when(func.col('merged_inst_id').isNull(), func.col('inst_id'))
                                                   .otherwise(func.col('merged_inst_id')))
                            .filter(func.col('merged_inst_id')==func.col('inst_id'))
                            .select(*[func.col(col).alias(col +'_s') for col in ['merged_inst_id','name','city','uni_pub','cnrs','type','main_topic','type_fr','secteur', 'acces_rce','date_first_idex', 'fusion_date']],
                                   func.concat_ws(';', func.transform(func.col('topic_share'), lambda x: func.to_json(x))).alias('topic_share_s'),
                                   *[ (func.col('type_fr') == type_).cast('int').alias(type_.lower().replace('é', 'e') +'_s')
                                         for type_ in ['Université', 'École']],
                                    *[(func.col('secteur') == type_).cast('int').alias(type_.lower().replace('é', 'e') +'_s')
                                     for type_ in ['public', 'privé']]
                                   )
                            , on = 'merged_inst_id_s', how ='left')

                    .withColumn('type_r', func.when(func.col('merged_inst_id_r')=="abroad", 'abroad')
                                         .when(func.col('merged_inst_id_r')=="exit", 'exit')
                                         .otherwise(func.col('type_r')))
                    .withColumn('type_s', func.when(func.col('merged_inst_id_s')=="abroad", 'abroad')
                                              .when(func.col('merged_inst_id_s')=="entry", 'entry').otherwise(func.col('type_s')))
                    .fillna(0, subset = ['uni_pub_r','uni_pub_s','cnrs_r','cnrs_s','universite_s','universite_r','ecole_r','ecole_s', 'prive_s','prive_r', 'public_s','public_r',
                                        "acces_rce_s",'date_first_idex_s', 'fusion_date_s', "acces_rce_r", 'date_first_idex_r', 'fusion_date_r'] )
                
                    
                    #.withColumn('size_r', func.max(func.col('stayers') * (func.col('merged_inst_id_r')==func.col('merged_inst_id_s')).cast('int'))
                    #
                    #            .over(Window.partitionBy('merged_inst_id_r','year')))
                    #
                    #.withColumn('size_s', func.max(func.col('stayers') * (func.col('merged_inst_id_r')==func.col('merged_inst_id_s')).cast('int'))
                    #                  .over(Window.partitionBy('merged_inst_id_s','year')))

                          )


In [61]:
panel_inst_flows_merged.write.mode('overwrite').parquet('file:\\' + save_path + 'inst_level_flows.parquet')

In [62]:
panel_inst_flows_merged = spark.read.parquet('file:\\' + save_path + 'inst_level_flows.parquet')

In [63]:
panel_inst_flows_merged.filter( (func.col('merged_inst_id_r')== "I4210097432" ) | (func.col('merged_inst_id_s')== "I4210097432") ).count()

7618

In [64]:
panel_inst_flows_merged.count() # 11_06_26 : 10508455

10508455

In [51]:
panel_inst_flows_merged.select('merged_inst_id_s','merged_inst_id_r').distinct().count()

426286

In [52]:
panel_inst_flows_merged.groupBy('merged_inst_id_s','merged_inst_id_r','domain','year').count().groupBy('count').count().show()

+-----+--------+
|count|   count|
+-----+--------+
|    1|10665002|
+-----+--------+



In [53]:
panel_inst_flows_merged.select('merged_inst_id_s','merged_inst_id_r').filter(func.col('merged_inst_id_s') != func.col('merged_inst_id_r')).distinct().count()

423608

In [54]:
panel_inst_flows_merged.select('merged_inst_id_s').distinct().count()

2943

In [55]:
panel_inst_flows_merged.select('merged_inst_id_r').distinct().count()

3021

In [56]:
panel_inst_flows_merged.filter(func.col('movers_w')>0).select('merged_inst_id_s','merged_inst_id_r').filter(func.col('merged_inst_id_s') != func.col('merged_inst_id_r')).distinct().count()

346629

# Institution-domain level for n_au varied

In [30]:
pre_inst_level_moves = (au_inst_level_plus_info
                    .withColumn('stayer', func.array_contains(func.col('inst_set_sender'), func.col('inst_id_receiver')).cast('int'))
                    .withColumn('period_total_inst', func.sum(1-func.col('stayer')).over(Window.partitionBy('author_id','inst_id_receiver').orderBy('year')))
                    .join(inst
                            .withColumnRenamed('inst_id','inst_id_sender')
                            , on = 'inst_id_sender', how ='left')
                           .withColumn('gender', func.when(func.col('gender')=="M", "men")
                                       .when(func.col('gender')=="F", 'women')
                                      )
                    .groupBy('author_id','year', func.col('inst_id_receiver').alias('inst_id'),'period_total_inst','gender')
                    .agg(func.max((func.col('inst_id_sender')=='abroad').cast('int')).alias('from_abroad'),
                         func.max((func.col('inst_id_sender')=='entry').cast('int')).alias('entrant'),
                         func.max('stayer').alias('stayer'),
                        *[func.first(col).alias(col) for col in ['entry_year', 'exit_year',
                                                                  'entry_inst_id']],
                           func.max('cnrs').alias('from_cnrs'),
                         *[func.max( ((func.col(col)!=0) & (func.col(col).isNotNull())) ).cast('int').alias('from_' +col)
                                    for col in ['acces_rce','date_first_idex','fusion_date']],
                         *[func.max( (func.col('type')== type_).cast('int')).alias('from_t_' + type_)
                           for type_ in ['company', 'education','healthcare','facility','government','archive','other']],
                         *[func.max( (func.col('secteur')==secteur).cast('int')).alias('from_' + secteur)
                           for secteur in ['public','privé']]
                        )
                    .join(au_prod_measures
                       .select('author_id',
                               *[func.when(func.col(col)<=0.25, 'Q1').when(func.col(col)<=0.5, 'Q2').when(func.col(col)<=0.75, 'Q3').otherwise('Q4').alias(col.replace('pct','quart'))
                                 for col in ['pct_cit','pct_cit_2y']])
                               
                               , on = 'author_id', how = 'left')
                    .join(df_au.select('author_id', func.element_at(func.col('topics'), 1).alias('topic'), 'summary_stats.h_index')
                       .select('author_id',  func.regexp_replace(func.col('topic.domain.id'), 'https://openalex.org/domains/', '').alias("domain"),
                              'h_index')
                       , on = 'author_id', how = 'left')
                    .withColumn('duration_period', func.col('year')-func.min(func.col('year')).over(Window.partitionBy('author_id','inst_id','period_total_inst')))
                    .join(weights_by_au_year, on = ['author_id','year'], how = 'left')
                   )


In [54]:
pre_inst_level_moves.cache()
pre_inst_level_moves.show()

+-----------+----+-----------+-----------------+------+-----------+-------+------+----------+---------+--------------------+---------+--------------+--------------------+----------------+--------------+----------------+-----------------+---------------+-----------------+--------------+------------+-----------+----------+---------+------------+------+-------+---------------+--------------------+
|  author_id|year|    inst_id|period_total_inst|gender|from_abroad|entrant|stayer|entry_year|exit_year|       entry_inst_id|from_cnrs|from_acces_rce|from_date_first_idex|from_fusion_date|from_t_company|from_t_education|from_t_healthcare|from_t_facility|from_t_government|from_t_archive|from_t_other|from_public|from_privé|quart_cit|quart_cit_2y|domain|h_index|duration_period|             weights|
+-----------+----+-----------+-----------------+------+-----------+-------+------+----------+---------+--------------------+---------+--------------+--------------------+----------------+--------------+--

In [62]:
from itertools import product
inst_level = (pre_inst_level_moves
              .withColumn('junior', (func.col('year')-func.col('entry_year')<=5).cast('int'))
              .withColumn('senior', (func.col('year')-func.col('entry_year')>=15).cast('int'))
              .withColumn('medium', ((func.col('year')-func.col('entry_year')>5) & (func.col('year')-func.col('entry_year')<15)) .cast('int'))
              .withColumn('foreign_entrant', func.array_contains(func.col('entry_inst_id'), 'abroad').cast('int'))
              .groupBy('inst_id', 'domain','year')
              .agg( func.countDistinct('author_id').alias('total'),
                   func.sum('stayer').alias('stayers'),
                   func.sum(1-func.col('stayer')).alias('movers'),
                   
                   func.sum('weights').alias('total_w'),
                   func.sum(func.col('weights')*func.col('stayer')).alias('stayers_w'),
                   func.sum(func.col('weights')*(1-func.col('stayer'))).alias('movers_w'),


                   func.sum( (func.col('year') == func.col('exit_year')).cast('int') ).alias('exits'),
                   func.sum(func.col('weights')*(func.col('year') == func.col('exit_year')).cast('int') ).alias('exits_w'),

                   # By category of author

                           *[func.sum(category).alias('total_' + category )
                             for category in ['junior','senior','medium', 'foreign_entrant','entrant'] +
                             [col for col in pre_inst_level_moves.columns if 'from_' in col] 
                           ],

                           *[func.sum(func.col('stayer')*func.col(category)).alias('stayers_' + category) for category in 
                             ['junior','senior','medium', 'foreign_entrant'] +
                             [col for col in pre_inst_level_moves.columns if 'from_' in col]
                           ],

                           *[func.sum( (1-func.col('stayer'))*func.col(category)).alias('movers_' + category) 
                             for category in ['junior','senior','medium', 'foreign_entrant','entrant'] +
                             [col for col in pre_inst_level_moves.columns if 'from_' in col]
                           ],
                           
                           *[func.sum(func.col(category)*(func.col('exit_year')==func.col('year')).cast('int') ).alias('exits_' + category)for category in 
                             ['junior','senior','medium', 'foreign_entrant'] +
                             [col for col in pre_inst_level_moves.columns if 'from_' in col]
                           ],

                        
                           *[func.sum(func.col('weights')*func.col(category)).alias('total_w_' + category)for category in 
                             ['junior','senior','medium', 'foreign_entrant'] +
                             [col for col in pre_inst_level_moves.columns if 'from_' in col]
                           ],

                           *[func.sum(func.col('weights')*func.col(category)*(func.col('exit_year')==func.col('year')).cast('int') ).alias('exits_w_' + category)for category in 
                             ['junior','senior','medium',  'foreign_entrant'] +
                             [col for col in pre_inst_level_moves.columns if 'from_' in col]
                           ],

                           *[func.sum(func.col('weights')*func.col('stayer')*func.col(category)).alias('stayers_w_' + category) 
                             for category in ['junior','senior','medium', 'foreign_entrant'] +
                             [col for col in pre_inst_level_moves.columns if 'from_' in col]
                           ],
                           *[func.sum(func.col('weights')*(1-func.col('stayer'))*func.col(category)).alias('movers_w_' + category) 
                             for category in ['junior','senior','medium', 'foreign_entrant','entrant'] +
                             [col for col in pre_inst_level_moves.columns if 'from_' in col]
                           ],

                #By time since arrival in institution

                
                           *[func.sum((func.col('duration_period')<= duration ).cast('int')).alias('total_under_' + str(duration) + 'y' )
                             for duration in [2,5,10]
                           ],

                           *[func.sum(func.col('stayer')*(func.col('duration_period')<= duration ).cast('int')).alias('stayers_under_' + str(duration) + 'y')
                             for duration in [2,5,10]
                           ],
                        
                           *[func.sum(func.col('weights')*(func.col('duration_period')<= duration ).cast('int')).alias('total_w_under_' + str(duration) + 'y')
                             for duration in [2,5,10]
                           ],

                           *[func.sum(func.col('weights')*(func.col('duration_period')<= duration ).cast('int')*(func.col('exit_year')==func.col('year')).cast('int') ).alias('exits_w_under_' + str(duration) + 'y')
                             for duration in [2,5,10]
                           ],

                           *[func.sum(func.col('weights')*func.col('stayer')*(func.col('duration_period')<= duration ).cast('int')).alias('stayers_w_under_' + str(duration) + 'y') 
                             for duration in [2,5,10]
                           ],
                           *[func.sum(func.col('weights')*(1-func.col('stayer'))*(func.col('duration_period')<= duration ).cast('int')).alias('movers_w_under_' + str(duration) + 'y') 
                             for duration in [2,5,10]
                           ],

                   #By quartile of productivity
                   
                           *[func.sum((func.col(prod)==quartile).cast('int') ).alias('total_' + quartile + '_' + prod.replace('quart', '') )
                             for quartile, prod in product(['Q1','Q2','Q3','Q4'], ['quart_cit','quart_cit_2y'])
                           ],
                           *[func.sum((1-func.col('stayer'))*(func.col(prod)==quartile).cast('int') ).alias('movers_' + quartile + '_' + prod.replace('quart', '') )
                             for quartile, prod in product(['Q1','Q2','Q3','Q4'], ['quart_cit','quart_cit_2y'])
                           ],
                           *[func.sum((func.col('year')==func.col('exit_year')).cast('int')*(func.col(prod)==quartile).cast('int') ).alias('exits_' + quartile + '_' + prod.replace('quart', '') )
                             for quartile, prod in product(['Q1','Q2','Q3','Q4'], ['quart_cit','quart_cit_2y'])
                           ],

                           *[func.sum(func.col('weights')*(func.col(prod)==quartile).cast('int') ).alias('total_w_' + quartile + '_' + prod.replace('quart', '') )
                             for quartile, prod in product(['Q1','Q2','Q3','Q4'], ['quart_cit','quart_cit_2y'])
                           ],
                           *[func.sum(func.col('weights')*(1-func.col('stayer'))*(func.col(prod)==quartile).cast('int') ).alias('movers_w_' + quartile + '_' + prod.replace('quart', '') )
                             for quartile, prod in product(['Q1','Q2','Q3','Q4'], ['quart_cit','quart_cit_2y'])
                           ],
                           *[func.sum(func.col('weights')*(func.col('year')==func.col('exit_year')).cast('int')*(func.col(prod)==quartile).cast('int') ).alias('exits_w_' + quartile + '_' + prod.replace('quart', '') )
                             for quartile, prod in product(['Q1','Q2','Q3','Q4'], ['quart_cit','quart_cit_2y'])
                           ],

                   #By gender
#
                           *[func.sum((func.col('gender')==gender).cast('int') ).alias('total_' + gender )
                             for gender in ['men','women']
                           ],
                           *[func.sum((1-func.col('stayer'))*(func.col("gender")==gender).cast('int') ).alias('movers_' + gender  )
                             for gender in ['men','women']
                           ],
                           *[func.sum((func.col(cat)*(func.col("gender")==gender).cast('int') )).alias('total_' + gender + '_' + cat )
                             for cat, gender in product(['entrant', 'from_abroad'], ['men','women'])
                           ],

                           *[func.sum(func.col('weights')*(func.col('gender')==gender).cast('int') ).alias('total_w_' + gender  )
                             for gender in ['men','women']
                           ],
                           *[func.sum(func.col('weights')*(1-func.col('stayer'))*(func.col("gender")==gender).cast('int') ).alias('movers_w_' + gender )
                             for gender in ['men','women']
                           ],
                           *[func.sum(func.col('weights')*(func.col(cat)*(func.col("gender")==gender).cast('int') )).alias('total_w_' + gender + '_' + cat )
                             for cat, gender in product(['entrant', 'from_abroad'], ['men','women'])
                           ],



                  )
             )

                   
                   

In [63]:
inst_level.show()

+-----------+------+----+-----+-------+------+------------------+------------------+------------------+-----+------------------+------------+------------+------------+---------------------+-------------+-----------------+---------------+--------------------+--------------------------+----------------------+--------------------+----------------------+-----------------------+---------------------+-----------------------+--------------------+------------------+-----------------+----------------+--------------+--------------+--------------+-----------------------+-------------------+-----------------+----------------------+----------------------------+------------------------+----------------------+------------------------+-------------------------+-----------------------+-------------------------+----------------------+--------------------+-------------------+------------------+-------------+-------------+-------------+----------------------+--------------+------------------+-------------

In [64]:
inst_level.write.mode('overwrite').parquet('file:\\' + save_path + 'intermediary\\nondyad_inst_level_flows.parquet')

In [65]:
inst_level = spark.read.parquet('file:\\' + save_path + 'intermediary\\nondyad_inst_level_flows.parquet')

In [94]:
all_y_inst = (inst_level
              .groupBy('inst_id','domain')
              .agg(func.min(func.col('year').alias('first_year_lab')),
                   func.sequence(func.min('year'), func.lit(2020)).alias('year'))
               .withColumn('year', func.explode(func.col('year')))
             )

              

In [70]:
works_au_af_subset_winfo = (inst
                            .join(works_au_af
                                  .withColumnRenamed('publication_year','year'),
                                  on = ['inst_id'], how = 'left')
                            .join(journals_ranking, on = ['source_id','year'], how = 'left')
                            .join(pre_inst_level_moves, on = ['author_id','year','inst_id'], how = 'left')
                            .join(papers_textual_metrics, on = 'work_id', how = 'left')
              
                            
                            .withColumn('junior', (func.col('year')-func.col('entry_year')<=5).cast('int'))
                            .withColumn('senior', (func.col('year')-func.col('entry_year')>=15).cast('int'))
                            .withColumn('medium', ((func.col('year')-func.col('entry_year')>5) & (func.col('year')-func.col('entry_year')<15)) .cast('int'))
                            .withColumn('foreign_entrant', func.array_contains(func.col('entry_inst_id'), 'abroad').cast('int'))

                           )


In [71]:
inst_work_level = (works_au_af_subset_winfo
                   .groupBy('inst_id','year','work_id','domain')
                   .agg( *[func.first(col).alias(col)
                           for col in ['citations']
                           + [col for col in papers_textual_metrics.columns if col != 'work_id']],
                        
                        func.max((func.col('rank_source_pct')<= 0.5).cast('int')).alias('is_source_btm_50pct'),
                        func.max((func.col('rank_source_pct').between(0.5,0.9)).cast('int')).alias('is_source_mid_40pct'),
                        func.max((func.col('rank_source_pct')>= 0.8).cast('int')).alias('is_source_top_20pct'),
                        func.max((func.col('rank_source_pct')>= 0.9).cast('int')).alias('is_source_top_10pct'),
                        func.max((func.col('rank_source_pct')>= 0.95).cast('int')).alias('is_source_top_5pct'),
                        
                        func.countDistinct('author_id').alias('nr_coau'),
                        func.sum('weights').alias('nr_coau_w'),

                        *[func.max(category).alias('has_coau_' + category)
                          for category in ['from_abroad','entrant','junior','senior','medium','foreign_entrant']],
                        
                        *[func.max((func.col('duration_period')<=duration).cast('int') ).alias('has_coau_under_' + str(duration) + 'y'  )
                          for  duration in[2,5,10]],

                        *[func.max(category*(func.col('duration_period')<=duration).cast('int') ).alias('has_coau_' + category + '_under_' + str(duration) + 'y'  )
                          for category, duration in product(['from_abroad','entrant','junior','senior','medium','foreign_entrant'],[2,5,10])
                         ]
                       )
                  )

In [78]:
subset_outcomes_crossed = ['citations', 
                                            'new_word','new_phrase','new_word_comb_reuse','new_phrase_comb_reuse','semantic_distance',
                                            'is_source_btm_50pct','is_source_mid_40pct','is_source_top_20pct','is_source_top_10pct','is_source_top_5pct']
subset_cats_crossed = [ 'has_coau_entrant','has_coau_junior','has_coau_senior','has_coau_medium',
                       'has_coau_foreign_entrant','has_coau_under_2y','has_coau_under_5y','has_coau_under_10y',
                        'has_coau_foreign_entrant_under_2y','has_coau_foreign_entrant_under_5y','has_coau_foreign_entrant_under_10y']

In [87]:
inst_y_level_prod = (inst_work_level
                     .groupBy('inst_id','year', 'domain')
                     .agg(func.countDistinct('work_id').alias('publications'),
                          *[func.sum(col).alias(col.replace('has_','nr_' ).replace('is_', 'nr_'))
                            for col in inst_work_level.columns if col not in ['work_id','year','inst_id','domain', 'nr_coau','nr_coau_w'] ],

                          *[func.mean(col).alias(col) for col in ['nr_coau', 'nr_coau_w']],
                          
                          *[func.sum(func.col(outcome) * func.col(cat) ).alias(outcome.replace('is_', 'nr_') + '_' +
                                                                               cat.replace('has_','' ))
                            for outcome, cat in product(subset_outcomes_crossed, subset_cats_crossed) ],
                            

                            )
                    )

In [88]:
inst_y_level_prod.printSchema()

root
 |-- inst_id: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- domain: string (nullable = true)
 |-- publications: long (nullable = false)
 |-- citations: long (nullable = true)
 |-- new_word: double (nullable = true)
 |-- new_word_reuse: double (nullable = true)
 |-- new_phrase: double (nullable = true)
 |-- new_phrase_reuse: double (nullable = true)
 |-- new_word_comb: double (nullable = true)
 |-- new_word_comb_reuse: double (nullable = true)
 |-- new_phrase_comb: double (nullable = true)
 |-- new_phrase_comb_reuse: double (nullable = true)
 |-- semantic_distance: double (nullable = true)
 |-- nr_source_btm_50pct_raw: long (nullable = true)
 |-- nr_source_mid_40pct_raw: long (nullable = true)
 |-- nr_source_top_20pct_raw: long (nullable = true)
 |-- nr_source_top_10pct_raw: long (nullable = true)
 |-- nr_source_top_5pct_raw: long (nullable = true)
 |-- nr_coau_from_abroad: long (nullable = true)
 |-- nr_coau_entrant: long (nullable = true)
 |-- nr_coau_junior:

In [89]:
inst_y_level_prod.write.mode('overwrite').parquet('file:\\' + save_path + 'intermediary\\nondyad_inst_level_prod.parquet')

In [91]:
inst_y_level_prod = spark.read.parquet('file:\\' + save_path + 'intermediary\\nondyad_inst_level_prod.parquet')

In [92]:
inst_y_level_prod.show()

+-----------+----+------+------------+---------+--------+--------------+----------+----------------+-------------+-------------------+---------------+---------------------+------------------+-----------------------+-----------------------+-----------------------+-----------------------+----------------------+-------------------+---------------+--------------+--------------+--------------+-----------------------+----------------+----------------+-----------------+----------------------------+----------------------------+-----------------------------+------------------------+------------------------+-------------------------+-----------------------+-----------------------+------------------------+-----------------------+-----------------------+------------------------+-----------------------+-----------------------+------------------------+--------------------------------+--------------------------------+---------------------------------+------------------+------------------+------------

In [95]:
inst_level_all = (all_y_inst
                  .join(inst, on ='inst_id', how = 'left')
                  .join(inst_y_level_prod, on = ['inst_id','domain','year'], how = 'left')
                  .join(inst_level, on = ['inst_id','domain','year'], how = 'left')
                  .fillna(0, subset = ['acces_rce','date_first_idex','fusion_date']+
                          [col for col in inst_level.columns + inst_y_level_prod.columns
                                       if col not in ['inst_id','domain','year']])
                  .join(budget_inst_matched, on = ['inst_id','year'], how = 'left')
                 )
                  

In [8]:
inst_level_all.write.mode('overwrite').parquet('file:\\' + save_path + 'inst_field_year_panel_mob_prod.parquet')

In [5]:
inst_level_all = spark.read.parquet('file:\\' + save_path + 'inst_field_year_panel_mob_prod_wo_budget.parquet')

In [7]:
inst_level_all = (inst_level_all.join(budget_inst_matched,on = ['inst_id','year'], how = 'left'))

In [102]:
save_path+"inst_field_year_panel_mob_prod.parquet"

'D:\\panel_fr_res\\data\\inst_field_year_panel_mob_prod.parquet'

In [98]:
inst_level_all.show()

+----------+------+----+---------------------------+--------------------+---------+------------+--------------------+--------------------+---------------+--------------------+-------+------------+----+--------------------+-------------------+-------------------+-------+---------+---------------+-----------+--------------+------------+---------+--------+--------------+----------+----------------+-------------+-------------------+---------------+---------------------+------------------+-----------------------+-----------------------+-----------------------+-----------------------+----------------------+-------------------+---------------+--------------+--------------+--------------+-----------------------+----------------+----------------+-----------------+----------------------------+----------------------------+-----------------------------+------------------------+------------------------+-------------------------+-----------------------+-----------------------+-----------------------

In [9]:
spark.catalog.clearCache()
spark.stop()